# Corpus Integration — merging supplementary batches

Merges later collection batches into the main corpus.

**Why it re-runs deduplication:** top-up scrapes overwhelmingly return images already collected.
Without re-deduplicating at merge time, each new batch would reintroduce duplicates that the earlier
pass had already removed.


In [ ]:
import shutil

In [ ]:
import os
from google.colab import drive

In [ ]:
# Step 1: Mount Google Drive
drive.mount('/content/drive')


In [ ]:
root_folder = '/content/drive/MyDrive/LGBTQ Memes/Additionally scraped images'

In [ ]:
# prompt: traverse three subfolders in root_folder and rename each image as "add_{count}" where count is initiated as 1 and incremented by for each image that you traverse in the folder.


count = 1
for subdir, dirs, files in os.walk(root_folder):
  for file in files:
    if file.lower().endswith(('.png', '.jpg', '.jpeg', '.gif', '.bmp', '.webp')): # Add more extensions if needed
      src_path = os.path.join(subdir, file)
      dst_path = os.path.join(subdir, f"add_{count}{os.path.splitext(file)[1]}")
      try:
        os.rename(src_path, dst_path)
        print(f"Renamed '{src_path}' to '{dst_path}'")
        count += 1
      except OSError as e:
        print(f"Error renaming '{src_path}': {e}")


In [ ]:
# prompt: once again traverse the three sub folders in the root_folder, and copy each image into a new folder named 'combined'

import shutil
import os

combined_folder = os.path.join(root_folder, 'combined')

# Create the combined folder if it doesn't exist
os.makedirs(combined_folder, exist_ok=True)

for subdir, dirs, files in os.walk(root_folder):
  for file in files:
    if file.lower().endswith(('.png', '.jpg', '.jpeg', '.gif', '.bmp', '.webp')):
      src_path = os.path.join(subdir, file)
      dst_path = os.path.join(combined_folder, file)
      try:
        shutil.copy2(src_path, dst_path) # copy2 preserves metadata
        print(f"Copied '{src_path}' to '{dst_path}'")
      except shutil.SameFileError:
        print(f"Skipping '{src_path}' as it's the same as '{dst_path}'")
      except OSError as e:
        print(f"Error copying '{src_path}': {e}")


In [ ]:
pip install imagededup opencv-python matplotlib

In [ ]:
import cv2
import matplotlib.pyplot as plt
from imagededup.methods import PHash

In [ ]:
dataset_path = '/content/drive/MyDrive/LGBTQ Memes/Additionally scraped images/combined'

# Initialize the PHash model
phash = PHash()

# Generate hashes for all images
encodings = phash.encode_images(image_dir=dataset_path)

# Find duplicates with a similarity threshold
duplicates = phash.find_duplicates(encoding_map=encodings, max_distance_threshold=5)

In [ ]:
# Track images to delete (keep the first occurrence)
deleted_files = set()

for original, dup_list in duplicates.items():
    for duplicate in dup_list:
        duplicate_path = os.path.join(dataset_path, duplicate)

        if os.path.exists(duplicate_path) and duplicate not in deleted_files:
            os.remove(duplicate_path)  # Delete duplicate
            deleted_files.add(duplicate)  # Mark as deleted

print(f"✅ Duplicate images removed from: {dataset_path}")
print(f"📁 Unique images are now stored in: {dataset_path}")

In [ ]:
# Generate hashes for all images in the new folder
encodings = phash.encode_images(image_dir=dataset_path)

# Find duplicates
duplicates = phash.find_duplicates(encoding_map=encodings, max_distance_threshold=5)

In [ ]:
# Count duplicates
total_duplicates = sum(len(dup_list) for dup_list in duplicates.values())

# Function to display duplicates side by side
def display_duplicates(pairs):
    for original, duplicate in pairs:
        try:
            img1 = cv2.imread(os.path.join(unique_folder, original))
            img2 = cv2.imread(os.path.join(unique_folder, duplicate))

            if img1 is None or img2 is None:
                print(f"Could not load images: {original}, {duplicate}")
                continue

            img1 = cv2.cvtColor(img1, cv2.COLOR_BGR2RGB)
            img2 = cv2.cvtColor(img2, cv2.COLOR_BGR2RGB)

            # Display images side by side
            fig, ax = plt.subplots(1, 2, figsize=(10, 5))
            ax[0].imshow(img1)
            ax[0].set_title(f"Original\n{original}")
            ax[0].axis('off')

            ax[1].imshow(img2)
            ax[1].set_title(f"Duplicate\n{duplicate}")
            ax[1].axis('off')

            plt.show()

        except Exception as e:
            print(f"Error displaying images: {e}")

# Print duplicate pairs
if total_duplicates > 0:
    print(f"\n=== Found {total_duplicates} Duplicate Images ===")
    duplicate_pairs = []

    for orig, dup_list in duplicates.items():
        for dup in dup_list:
            print(f"Original: {orig}  ||  Duplicate: {dup}")
            duplicate_pairs.append((orig, dup))

    # Display duplicates
    display_duplicates(duplicate_pairs)
else:
    print("\n✅ No duplicates found!")

print("\nDuplicate detection complete! Review images before manual deletion.")

In [ ]:
source_folder = '/content/drive/MyDrive/LGBTQ Memes/Additionally scraped images/combined'
dest_folder = '/content/drive/MyDrive/LGBTQ Memes/Combined_1500/unique_images'

In [ ]:
# prompt: copy all the images from source_folder to dest_folder

import shutil
import os

def copy_images(source_folder, dest_folder):
    """Copies all images from the source folder to the destination folder."""

    # # Create the destination folder if it doesn't exist
    # os.makedirs(dest_folder, exist_ok=True)

    for filename in os.listdir(source_folder):
        if filename.lower().endswith(('.png', '.jpg', '.jpeg', '.gif', '.bmp', '.webp')):
            source_path = os.path.join(source_folder, filename)
            dest_path = os.path.join(dest_folder, filename)

            try:
                shutil.copy2(source_path, dest_path)  # copy2 preserves metadata
                print(f"Copied '{source_path}' to '{dest_path}'")
            except shutil.SameFileError:
                print(f"Skipping '{source_path}' as it's the same as '{dest_path}'")
            except OSError as e:
                print(f"Error copying '{source_path}': {e}")

# Example usage (replace with your actual folder paths)
source_folder = '/content/drive/MyDrive/LGBTQ Memes/Additionally scraped images/combined'
dest_folder = '/content/drive/MyDrive/LGBTQ Memes/Combined_1500/unique_images'
copy_images(source_folder, dest_folder)


In [ ]:
# Generate hashes for all images
encodings = phash.encode_images(image_dir=dest_folder)

# Find duplicates with a similarity threshold
duplicates = phash.find_duplicates(encoding_map=encodings, max_distance_threshold=5)

In [ ]:
# Count duplicates
total_duplicates = sum(len(dup_list) for dup_list in duplicates.values())

# Function to display duplicates side by side
def display_duplicates(pairs):
    for original, duplicate in pairs:
        try:
            img1 = cv2.imread(os.path.join(dest_folder, original))
            img2 = cv2.imread(os.path.join(dest_folder, duplicate))

            if img1 is None or img2 is None:
                print(f"Could not load images: {original}, {duplicate}")
                continue

            img1 = cv2.cvtColor(img1, cv2.COLOR_BGR2RGB)
            img2 = cv2.cvtColor(img2, cv2.COLOR_BGR2RGB)

            # Display images side by side
            fig, ax = plt.subplots(1, 2, figsize=(10, 5))
            ax[0].imshow(img1)
            ax[0].set_title(f"Original\n{original}")
            ax[0].axis('off')

            ax[1].imshow(img2)
            ax[1].set_title(f"Duplicate\n{duplicate}")
            ax[1].axis('off')

            plt.show()

        except Exception as e:
            print(f"Error displaying images: {e}")

# Print duplicate pairs
if total_duplicates > 0:
    print(f"\n=== Found {total_duplicates} Duplicate Images ===")
    duplicate_pairs = []

    for orig, dup_list in duplicates.items():
        for dup in dup_list:
            print(f"Original: {orig}  ||  Duplicate: {dup}")
            duplicate_pairs.append((orig, dup))

    # Display duplicates
    display_duplicates(duplicate_pairs)
else:
    print("\n✅ No duplicates found!")

print("\nDuplicate detection complete! Review images before manual deletion.")

In [ ]:
# Track images to delete (keep the first occurrence)
deleted_files = set()

for original, dup_list in duplicates.items():
    for duplicate in dup_list:
        duplicate_path = os.path.join(dataset_path, duplicate)

        if os.path.exists(duplicate_path) and duplicate not in deleted_files:
            os.remove(duplicate_path)  # Delete duplicate
            deleted_files.add(duplicate)  # Mark as deleted

print(f"✅ Duplicate images removed from: {dest_folder}")
print(f"📁 Unique images are now stored in: {dest_folder}")

In [ ]:
# Generate hashes for all images
encodings = phash.encode_images(image_dir=dest_folder)

# Find duplicates with a similarity threshold
duplicates = phash.find_duplicates(encoding_map=encodings, max_distance_threshold=5)

In [ ]:
# Count duplicates
total_duplicates = sum(len(dup_list) for dup_list in duplicates.values())

# Function to display duplicates side by side
def display_duplicates(pairs):
    for original, duplicate in pairs:
        try:
            img1 = cv2.imread(os.path.join(dest_folder, original))
            img2 = cv2.imread(os.path.join(dest_folder, duplicate))

            if img1 is None or img2 is None:
                print(f"Could not load images: {original}, {duplicate}")
                continue

            img1 = cv2.cvtColor(img1, cv2.COLOR_BGR2RGB)
            img2 = cv2.cvtColor(img2, cv2.COLOR_BGR2RGB)

            # Display images side by side
            fig, ax = plt.subplots(1, 2, figsize=(10, 5))
            ax[0].imshow(img1)
            ax[0].set_title(f"Original\n{original}")
            ax[0].axis('off')

            ax[1].imshow(img2)
            ax[1].set_title(f"Duplicate\n{duplicate}")
            ax[1].axis('off')

            plt.show()

        except Exception as e:
            print(f"Error displaying images: {e}")

# Print duplicate pairs
if total_duplicates > 0:
    print(f"\n=== Found {total_duplicates} Duplicate Images ===")
    duplicate_pairs = []

    for orig, dup_list in duplicates.items():
        for dup in dup_list:
            print(f"Original: {orig}  ||  Duplicate: {dup}")
            duplicate_pairs.append((orig, dup))

    # Display duplicates
    display_duplicates(duplicate_pairs)
else:
    print("\n✅ No duplicates found!")

print("\nDuplicate detection complete! Review images before manual deletion.")

In [ ]:
# Initialize PHash model
phash = PHash()

# Generate hashes for all images in the new folder
encodings = phash.encode_images(image_dir=dest_folder)

# Find duplicates
duplicates = phash.find_duplicates(encoding_map=encodings, max_distance_threshold=5)

# Track images to delete (keep the first occurrence)
deleted_files = set()

for original, dup_list in duplicates.items():
    for duplicate in dup_list:
        duplicate_path = os.path.join(dest_folder, duplicate)

        if os.path.exists(duplicate_path) and duplicate not in deleted_files:
            os.remove(duplicate_path)  # Delete duplicate
            deleted_files.add(duplicate)  # Mark as deleted

print(f"✅ Duplicate images removed from: {dest_folder}")
print(f"📁 Unique images are now stored in: {dest_folder}")

In [ ]:
# Count the number of files in the unique_folder
num_files = 0
if os.path.exists(dest_folder):
    num_files = len([f for f in os.listdir(dest_folder) if os.path.isfile(os.path.join(dest_folder, f))])
    print(f"Number of files in {dest_folder}: {num_files}")
else:
    print(f"Error: The folder '{dest_folder}' does not exist.")